In [ ]:
# --- Arranque del entorno local (en Google Colab no cambia nada) ---
import pathlib
import sys
import types

try:
    _raiz = next(
        d
        for d in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
        if (d / "curso_setup.py").exists()
    )
    sys.path.insert(0, str(_raiz))
    import curso_setup
except StopIteration:  # Google Colab: se usa un sustituto mínimo
    import subprocess

    def _clonar(destino="curso_IA_CHEC"):
        if not pathlib.Path(destino).is_dir():
            subprocess.run(
                ["git", "clone", "https://github.com/UN-GCPDS/curso_IA_CHEC.git", destino],
                check=True,
            )
        return pathlib.Path(destino)

    def _descargar(file_id, destino):
        if not pathlib.Path(destino).exists():
            import gdown

            gdown.download(id=file_id, output=destino, quiet=False)
        return pathlib.Path(destino)

    curso_setup = types.SimpleNamespace(
        en_colab=lambda: True,
        init=lambda *a, **k: pathlib.Path.cwd(),
        clonar_curso=_clonar,
        descargar_drive=_descargar,
    )

curso_setup.init()

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# =========================
# 1) Cargar California Housing (sklearn)
# =========================
data = fetch_california_housing()
X = data.data.astype("float32")   # (N, 8)
y = data.target.astype("float32") # (N,)

# =========================
# 2) Split train/test
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =========================
# 3) Escalado (sklearn)
# =========================
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype("float32")
X_test  = scaler.transform(X_test).astype("float32")

# =========================
# 4) Modelo denso (regresión)
# =========================
model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(128, activation="relu"),
    layers.Dense(64, activation="relu"),
    layers.Dense(1)  # salida lineal para regresión
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="mse",
    metrics=[keras.metrics.MeanAbsoluteError(name="mae")]
)

# =========================
# 5) Entrenamiento
# =========================
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_mae", patience=10, restore_best_weights=True
    )
]

history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=200,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

# =========================
# 6) Evaluación
# =========================
loss, mae = model.evaluate(X_test, y_test, verbose=0)
print(f"Test MAE: {mae:.4f}")

# (opcional) ejemplo de predicción
pred = model.predict(X_test[:5], verbose=0).ravel()
print("y_true:", y_test[:5])
print("y_pred:", pred)

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# =========================
# 1) Cargar MNIST
# =========================
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Normalizar a [0,1]
x_train = x_train.astype("float32") / 255.0
x_test  = x_test.astype("float32") / 255.0

# =========================
# 2) Modelo Denso (MLP)
# =========================
model = keras.Sequential([
    layers.Input(shape=(28, 28)),
    layers.Flatten(),                 # (28,28) -> (784,)
    layers.Dense(256, activation="relu"),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(10, activation="softmax")  # 10 clases
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# =========================
# 3) Entrenamiento
# =========================
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=3, restore_best_weights=True
    )
]

history = model.fit(
    x_train, y_train,
    validation_split=0.1,
    epochs=30,
    batch_size=128,
    callbacks=callbacks,
    verbose=1
)


# =========================
# 4) Evaluación
# =========================
loss, acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Test Accuracy: {acc:.4f}")
# (opcional) predicción rápida
pred = model.predict(x_test[:5], verbose=0).argmax(axis=1)
print("y_true:", y_test[:5])
print("y_pred:", pred)